# Fed-ISIC2019 6-Client 연합학습 — Kaggle 전체 실행 노트북
`NOWEAZ/fedisic-federated-framework` 브랜치 `fix/notebook-source-resume-state` 코드를 그대로 사용합니다.

**실행 전 설정(우측 Settings):** Accelerator = GPU (T4 x2 또는 P100), Internet = On.
**실행 방법:** 처음에는 위에서부터 한 셀씩 실행해 오류가 없는지 확인한 뒤, `Save Version → Save & Run All (Commit)`으로 전체를 돌리세요. 결과는 `/kaggle/working/results_6client`에 저장되고 Commit 후 Output 탭에서 받을 수 있습니다.

순서: 환경 → 코드/FLamby 설치 → 데이터 → Stage 1(시드 41, μ 탐색) → μ 자동 선정 → Stage 2(시드 42/100/2026, test는 `--final_test`로 1회) → 요약.

In [1]:
import os, subprocess, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
!nvidia-smi
assert torch.cuda.is_available(), "GPU가 꺼져 있습니다. Settings > Accelerator에서 GPU를 선택하세요."

torch 2.11.0+cu128 | cuda True
GPU: Tesla T4 | count: 2
Sun Nov  1 10:19:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                

In [2]:
# 1. 프로젝트 코드(수정 브랜치)와 FLamby(고정 커밋) 준비
REPO_URL = "https://github.com/NOWEAZ/fedisic-federated-framework.git"
REPO_BRANCH = "fix/notebook-source-resume-state"
PROJECT_DIR = "/kaggle/working/CDII"
FLAMBY_DIR = "/kaggle/working/FLamby"
FLAMBY_COMMIT = "edacf54d5211520583b0133d55ac39b6fda8324b"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -1
!pip install -q albumentations==1.3.1
!pip install -q -r requirements.txt
if not os.path.exists(FLAMBY_DIR):
    !git clone https://github.com/owkin/FLamby.git {FLAMBY_DIR}
    !cd {FLAMBY_DIR} && git checkout {FLAMBY_COMMIT}
!cd {FLAMBY_DIR} && pip install -q -e ".[isic2019]"
import sys
if FLAMBY_DIR not in sys.path: sys.path.insert(0, FLAMBY_DIR)
import flamby; print("FLamby:", flamby.__file__)

Cloning into '/kaggle/working/CDII'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 95 (delta 54), reused 92 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 90.98 KiB | 6.50 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/kaggle/working/CDII
e39f841 (HEAD -> fix/notebook-source-resume-state, origin/fix/notebook-source-resume-state) notebook: clone the fix branch by default
Cloning into '/kaggle/working/FLamby'...
remote: Enumerating objects: 6531, done.
remote: Counting objects: 100% (1597/1597), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 6531 (delta 1436), reused 1303 (delta 1303), pack-reused 4934 (from 1)
Receiving objects: 100% (6531/6531), 69.62 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (4163/4163), done.
Note: switching to 'edacf54d5211520583b0133d55ac39b6fda8324b'.

You are in 'detached HEAD' state. You can look around, 

In [3]:
# 2. 데이터 준비. 우선순위: (a) /kaggle/input 에 붙인 전처리 데이터셋 -> (b) ISIC2019 다운로드 + FLamby 전처리
#    (ISIC 2019 / HAM10000 데이터는 CC-BY-NC 라이선스입니다.)
import glob, shutil
SAVE_DIR = "/kaggle/working/results_6client"
DATA_PATH = "/kaggle/temp/isic_data"   # 임시 디스크(출력에 포함되지 않음, 용량 여유)
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(DATA_PATH, exist_ok=True)
PRE = "ISIC_2019_Training_Input_preprocessed"
PRE_DIR = os.path.join(DATA_PATH, PRE)

def have_data(): return os.path.isdir(PRE_DIR) and len(os.listdir(PRE_DIR)) > 20000

if not have_data():
    tars = glob.glob("/kaggle/input/**/*isic*preprocessed*.tar", recursive=True)
    dirs = glob.glob(f"/kaggle/input/**/{PRE}", recursive=True)
    if tars:
        print("extracting tar:", tars[0]); subprocess.run(["tar", "-xf", tars[0], "-C", DATA_PATH], check=True)
    elif dirs:
        print("입력 데이터셋 폴더를 복사합니다:", dirs[0]); shutil.copytree(dirs[0], PRE_DIR)
if not have_data():
    print("전처리 데이터가 없어 ISIC2019를 다운로드·전처리합니다 (약 9GB, 수십 분).")
    SCRIPT_DIR = os.path.join(FLAMBY_DIR, "flamby", "datasets", "fed_isic2019", "dataset_creation_scripts")
    %cd {SCRIPT_DIR}
    !sed -i 's#isic-challenge-data.s3.amazonaws.com/2019/#isic-archive.s3.amazonaws.com/challenges/2019/#g' download_isic.py
    !sed -i 's#"unzip #"unzip -q -o #' download_isic.py
    !yes y | python download_isic.py --output-folder "{DATA_PATH}"
    !python resize_images.py
    %cd {PROJECT_DIR}
print("전처리 이미지 수:", len(os.listdir(PRE_DIR)) if os.path.isdir(PRE_DIR) else 0, "(정상: 약 23,247)")
assert have_data(), "데이터 준비 실패"

전처리 데이터가 없어 ISIC2019를 다운로드·전처리합니다 (약 9GB, 수십 분).
/kaggle/working/FLamby/flamby/datasets/fed_isic2019/dataset_creation_scripts
Have you taken the time to read and accept the data terms on the original website, available at the following link: https://challenge.isic-archive.com/data/ ? | (y/n)


Saving license agreement
You may now proceed to download.

--2026-11-01 10:19:42--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 52.217.201.34, 16.15.212.14, 16.15.207.255, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|52.217.201.34|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9771618190 (9.1G) [application/zip]
Saving to: ‘/kaggle/temp/isic_data/ISIC_2019_Training_Input.zip’

/kaggle/temp/isic_d 100%[===================>]   9.10G   100MB/s    in 2m 2s   

2026-11-01 10:21:45 (76.3 MB/s) - ‘/kaggle/temp/isic_data/ISIC_2019_

In [4]:
# 3. 스모크 테스트: 6개 센터 로더와 학습/검증/테스트 표본 수
%cd {PROJECT_DIR}
from data_utils import get_dataloaders, BATCH_SIZE, LR, NUM_CLIENTS, NUM_EPOCHS_POOLED, get_nb_max_rounds
cl, cen, val, test = get_dataloaders(batch_size=BATCH_SIZE, num_workers=2, data_path=DATA_PATH, seed=42, val_fraction=0.1, split_seed=0)
for k in range(NUM_CLIENTS):
    print(k, len(cl[k].dataset), len(val.client_eval_loaders[k].dataset), len(test.client_eval_loaders[k].dataset))
print("BATCH", BATCH_SIZE, "LR", LR, "pooled epochs", NUM_EPOCHS_POOLED, "FL rounds", get_nb_max_rounds(100, batch_size=BATCH_SIZE))
# 기대값(train/val/test): C0 8937/993/2483, C1 2846/317/791, C2 2423/268/672, C3 1625/182/452, C4 589/66/164, C5 313/38/88
del cl, cen, val, test

/kaggle/working/CDII
0 8937 993 2483
1 2846 317 791
2 2423 268 672
3 1625 182 452
4 589 66 164
5 313 38 88
BATCH 64 LR 0.0005 pooled epochs 20 FL rounds 9


In [5]:
# 4. 공통 설정 (공식 스케줄)
from data_utils import BATCH_SIZE, LR, NUM_EPOCHS_POOLED, get_nb_max_rounds
NUM_UPDATES = 100
FL_ROUNDS = get_nb_max_rounds(NUM_UPDATES, batch_size=BATCH_SIZE)   # 9
CENTRAL_ROUNDS = NUM_EPOCHS_POOLED                                  # 20
CENTRAL_LOCAL_EPOCHS = 1
NUM_WORKERS = 4
EXPLORE_SEED = 41
FINAL_SEEDS = [42, 100, 2026]
MU_CANDIDATES = [0.001, 0.01, 0.1, 1.0]
COMMON_ARGS = (f'--batch_size {BATCH_SIZE} --lr {LR} --num_workers {NUM_WORKERS} --val_fraction 0.1 --split_seed 0 '
               f'--data_path "{DATA_PATH}" --save_dir "{SAVE_DIR}" --resume')
print(FL_ROUNDS, CENTRAL_ROUNDS, COMMON_ARGS)

9 20 --batch_size 64 --lr 0.0005 --num_workers 4 --val_fraction 0.1 --split_seed 0 --data_path "/kaggle/temp/isic_data" --save_dir "/kaggle/working/results_6client" --resume


In [6]:
# 5. Stage 1 (탐색 시드 41): FedAvg(mu=0)와 FedProx mu 후보. 같은 설정이 끝난 run은 --resume이 건너뜁니다.
%cd {PROJECT_DIR}
!python main.py --method fedavg --mu 0.0 --rounds {FL_ROUNDS} --num_updates {NUM_UPDATES} --seed {EXPLORE_SEED} {COMMON_ARGS}
for mu in MU_CANDIDATES:
    print(f"\n===== [Stage 1] FedProx mu={mu} seed={EXPLORE_SEED} =====")
    !python main.py --method fedprox --mu {mu} --rounds {FL_ROUNDS} --num_updates {NUM_UPDATES} --seed {EXPLORE_SEED} {COMMON_ARGS}

/kaggle/working/CDII
[Info] Using device: cuda | Seed: 41
[Info] Method: fedavg | Clients: 6 | Rounds: 9 | Num Updates/Round: 100 | Batch Size: 64 | LR: 0.0005 | Mu: 0.0
[Info] Number of clients: 6 | Client samples: [8937, 2846, 2423, 1625, 589, 313]
[Info] Centralized train samples: 16733 | Val samples: 1864 | Test samples: 4650 (val_fraction=0.1, split_seed=0)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 157MB/s]
[Info] Model Size: 15.49 MB | Comm Volume per Round (Down+Up): 185.85 MB
[Resume] 기존 체크포인트가 없어 처음부터 학습합니다.
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarnin

In [7]:
# 6. BEST_MU 자동 선정: validation macro-F1 마지막 3라운드 평균이 가장 높은 mu (동률이면 작은 mu)
import glob, json
import pandas as pd
rows = []
for p in sorted(glob.glob(os.path.join(SAVE_DIR, f"*_seed{EXPLORE_SEED}_metrics.csv"))):
    df = pd.read_csv(p)
    if df.empty or "mode" in df.columns: continue
    t = df.sort_values("round").tail(3)
    rows.append({"method": df["method"].iloc[0], "mu": float(df["mu"].iloc[0]), "rounds": int(df["round"].max()),
                 "val_macro_f1_last3": t["macro_f1"].mean(), "val_bal_acc_last3": t["balanced_acc"].mean(),
                 "val_f1_gap_last3": t["macro_f1_gap"].mean()})
s1 = pd.DataFrame(rows).sort_values("mu"); display(s1)
fp = s1[(s1.method == "fedprox") & (s1.rounds == FL_ROUNDS)]
assert len(fp) == len(MU_CANDIDATES), "Stage 1 FedProx run이 모두 끝나지 않았습니다."
BEST_MU = float(fp.sort_values(["val_macro_f1_last3", "mu"], ascending=[False, True]).iloc[0]["mu"])
print("BEST_MU =", BEST_MU)
json.dump({"best_mu": BEST_MU, "rule": "max val macro-F1 (last 3 rounds mean), tie -> smaller mu"}, open(os.path.join(SAVE_DIR, "best_mu.json"), "w"))
s1.to_csv(os.path.join(SAVE_DIR, "stage1_summary.csv"), index=False)

,method,mu,rounds,val_macro_f1_last3,val_bal_acc_last3,val_f1_gap_last3
0,fedavg,0.000,9,0.501817,0.623955,0.114194
1,fedprox,0.001,9,0.487539,0.620285,0.172018
2,fedprox,0.010,9,0.499516,0.627974,0.138652
3,fedprox,0.100,9,0.529871,0.654133,0.171564
4,fedprox,1.000,9,0.509393,0.630847,0.211380


BEST_MU = 0.1


In [8]:
# 7. Stage 2 (시드 42, 100, 2026): Centralized, FedAvg, FedProx(BEST_MU). 학습 후 test를 --final_test로 1회만 평가
%cd {PROJECT_DIR}
FINAL_ARGS = f"{COMMON_ARGS} --final_test"
for s in FINAL_SEEDS:
    print(f"\n===== [Stage 2] Centralized seed={s} =====")
    !python main.py --method centralized --rounds {CENTRAL_ROUNDS} --local_epochs {CENTRAL_LOCAL_EPOCHS} --seed {s} {FINAL_ARGS}
    print(f"\n===== [Stage 2] FedAvg seed={s} =====")
    !python main.py --method fedavg --mu 0.0 --rounds {FL_ROUNDS} --num_updates {NUM_UPDATES} --seed {s} {FINAL_ARGS}
    print(f"\n===== [Stage 2] FedProx mu={BEST_MU} seed={s} =====")
    !python main.py --method fedprox --mu {BEST_MU} --rounds {FL_ROUNDS} --num_updates {NUM_UPDATES} --seed {s} {FINAL_ARGS}

/kaggle/working/CDII

===== [Stage 2] Centralized seed=42 =====
[Info] Using device: cuda | Seed: 42
[Info] Method: centralized | Clients: 6 | Rounds: 20 | Local Epochs/Round: 1 | Batch Size: 64 | LR: 0.0005 | Mu: 0.0
[Info] Number of clients: 6 | Client samples: [8937, 2846, 2423, 1625, 589, 313]
[Info] Centralized train samples: 16733 | Val samples: 1864 | Test samples: 4650 (val_fraction=0.1, split_seed=0)
[Info] Model Size: 15.49 MB | Comm Volume per Round (Down+Up): 0.00 MB
[Resume] 기존 체크포인트가 없어 처음부터 학습합니다.
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
[Round 001/020] Train Loss: 7.4240 | Val Loss: 6.2580 | Bal Acc: 0.5757 (Std: 0.1499, Gap: 0.4153) | Mac

In [9]:
# 8. 요약: 결과 파일 목록과 test 요약, 하나로 묶기
import glob, json
print(sorted(os.listdir(SAVE_DIR)))
rows = []
for p in sorted(glob.glob(os.path.join(SAVE_DIR, "*_final_test.json"))):
    d = json.load(open(p)); rows.append({"file": os.path.basename(p), **{k: v for k, v in d.items() if isinstance(v, (int, float, str))}})
display(pd.DataFrame(rows))
!cd /kaggle/working && tar -czf results_6client.tar.gz results_6client && ls -la results_6client.tar.gz

['best_centralized_seed100.pt', 'best_centralized_seed2026.pt', 'best_centralized_seed42.pt', 'best_fedavg_seed100.pt', 'best_fedavg_seed2026.pt', 'best_fedavg_seed41.pt', 'best_fedavg_seed42.pt', 'best_fedprox_mu0.001_seed41.pt', 'best_fedprox_mu0.01_seed41.pt', 'best_fedprox_mu0.1_seed100.pt', 'best_fedprox_mu0.1_seed2026.pt', 'best_fedprox_mu0.1_seed41.pt', 'best_fedprox_mu0.1_seed42.pt', 'best_fedprox_mu1_seed41.pt', 'best_mu.json', 'centralized_seed100_env.json', 'centralized_seed100_final_test.json', 'centralized_seed100_final_test_preds.npz', 'centralized_seed100_metrics.csv', 'centralized_seed2026_env.json', 'centralized_seed2026_final_test.json', 'centralized_seed2026_final_test_preds.npz', 'centralized_seed2026_metrics.csv', 'centralized_seed42_env.json', 'centralized_seed42_final_test.json', 'centralized_seed42_final_test_preds.npz', 'centralized_seed42_metrics.csv', 'fedavg_seed100_env.json', 'fedavg_seed100_final_test.json', 'fedavg_seed100_final_test_preds.npz', 'fedavg_s

,file,eval_split,final_round
0,centralized_seed100_final_test.json,test,20
1,centralized_seed2026_final_test.json,test,20
2,centralized_seed42_final_test.json,test,20
3,fedavg_seed100_final_test.json,test,9
4,fedavg_seed2026_final_test.json,test,9
5,fedavg_seed42_final_test.json,test,9
6,fedprox_mu0.1_seed100_final_test.json,test,9
7,fedprox_mu0.1_seed2026_final_test.json,test,9
8,fedprox_mu0.1_seed42_final_test.json,test,9


-rw-r--r-- 1 root root 514125232 Nov  1 17:31 results_6client.tar.gz
